# 🚀 AIC-2026: Fast DAM & Audio ASR Embedding on Kaggle GPU

This notebook uses **NVIDIA GPU (CUDA FP16)** to embed:
1. **DAM Object Captions** (435,713 objects with bounding boxes) $\rightarrow$ **`BAAI/bge-m3`** (1024-d Dense).
2. **Audio ASR Speech Transcripts** (55,168 segments) $\rightarrow$ **`BAAI/bge-m3`** (1024-d Dense).

*(Note: SigLIP-2 visual feature embeddings are already extracted as `.f16.npy` files and excluded from this notebook).*

Estimated total execution time: **~10–12 minutes** on a single Kaggle T4 / P100 GPU.

## 📦 Step 1: Install Dependencies & Check GPU

In [ ]:
!nvidia-smi
!pip install -q qdrant-client transformers torch tqdm accelerate
!apt-get update -qq && apt-get install -y -qq rclone zip unzip

## 🔑 Step 2: Load Rclone Config from Kaggle Secret (`RCLONE_CONFIG_GDRIVE`)

In [ ]:
import os, re
from pathlib import Path
from kaggle_secrets import UserSecretsClient

# 1. Install rclone binary
!apt-get update -qq && apt-get install -y -qq rclone

# 2. Write rclone config from Secret
try:
    raw_secret = UserSecretsClient().get_secret('RCLONE_CONFIG_GDRIVE').strip()
    if not raw_secret.startswith('['):
        raw_secret = '[gdrive] ' + raw_secret
    formatted = raw_secret
    if '\n' not in formatted:
        formatted = formatted.replace('[gdrive]', '[gdrive]\n')
        formatted = re.sub(r'\s+(type\s*=)', r'\n\1', formatted)
        formatted = re.sub(r'\s+(scope\s*=)', r'\n\1', formatted)
        formatted = re.sub(r'\s+(token\s*=)', r'\n\1', formatted)
        formatted = re.sub(r'\s+(client_id\s*=)', r'\n\1', formatted)
        formatted = re.sub(r'\s+(client_secret\s*=)', r'\n\1', formatted)
    for path_str in ['/root/.config/rclone/rclone.conf', '/root/.rclone.conf']:
        p = Path(path_str)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(formatted, encoding='utf-8')
    print('✓ rclone.conf written successfully!')
except Exception as exc:
    print(f'❌ Error: {exc}')

# Test rclone connection
!rclone listremotes
!rclone lsd "gdrive:"

## 📥 Step 3: Download Raw DAM & ASR Files from Google Drive

In [ ]:
!mkdir -p /kaggle/working/data/dam_descriptions
!mkdir -p /kaggle/working/data/asr_segments
!mkdir -p /kaggle/working/data/map-keyframes

print("📥 Downloading raw files from Google Drive (gdrive:AIC_HCM)...\n")

# Fast multi-threaded sync
!rclone copy "gdrive:AIC_HCM/artifacts/dam_descriptions" /kaggle/working/data/dam_descriptions --transfers 16 -P
!rclone copy "gdrive:AIC_HCM/artifacts/asr_segments" /kaggle/working/data/asr_segments --transfers 16 -P
!rclone copy "gdrive:AIC_HCM/map-keyframes" /kaggle/working/data/map-keyframes --transfers 16 -P

import glob
print(f"\n✅ Download Verification:")
print(f"  • DAM JSONLs: {len(glob.glob('/kaggle/working/data/dam_descriptions/*.jsonl'))} files")
print(f"  • ASR Files:  {len(glob.glob('/kaggle/working/data/asr_segments/*.*'))} files")
print(f"  • Map CSVs:   {len(glob.glob('/kaggle/working/data/map-keyframes/*.csv'))} files")

## ⚡ Step 4: Write & Execute Embedding Runner (`run_kaggle_embedding.py`)

In [ ]:
%%writefile run_kaggle_embedding.py
import argparse
import csv
import json
import logging
import time
from pathlib import Path
from typing import Any, Generator

import numpy as np
import torch
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

class BGEFastEmbedder:
    def __init__(self, model_id: str = "BAAI/bge-m3"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        logger.info(f"⚡ Loading BGE-M3 in FP16 on {self.device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.model = AutoModel.from_pretrained(model_id, torch_dtype=dtype).to(self.device)
        self.model.eval()

    @torch.no_grad()
    def embed_batch(self, texts: list[str], batch_size: int = 128) -> np.ndarray:
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = [t if (t and isinstance(t, str)) else " " for t in texts[i : i + batch_size]]
            inputs = self.tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(self.device)
            outputs = self.model(**inputs)
            cls_repr = outputs.last_hidden_state[:, 0]
            normalized = torch.nn.functional.normalize(cls_repr, p=2, dim=1)
            all_embeddings.append(normalized.cpu().to(torch.float32).numpy())
        return np.vstack(all_embeddings) if all_embeddings else np.empty((0, 1024), dtype=np.float32)

def load_map_keyframes(map_dir: Path) -> dict[str, list[dict[str, Any]]]:
    video_to_keyframes = {}
    for csv_file in sorted(list(map_dir.glob("*.csv"))):
        video_id = csv_file.stem
        rows = []
        with open(csv_file, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                rows.append({
                    "n": int(row.get("n", 0)),
                    "pts_time": float(row.get("pts_time", 0.0)),
                    "fps": float(row.get("fps", 25.0)),
                    "frame_idx": int(row.get("frame_idx", 0)),
                })
        video_to_keyframes[video_id] = sorted(rows, key=lambda x: x["pts_time"])
    return video_to_keyframes

def chunk_list(items: list[Any], chunk_size: int = 1000) -> Generator[list[Any], None, None]:
    for i in range(0, len(items), chunk_size):
        yield items[i : i + chunk_size]

def process_dam_descriptions(dam_dir: Path, embedder: BGEFastEmbedder, client: QdrantClient, batch_size: int = 128):
    logger.info("🎯 STEP 1: PROCESSING DAM OBJECT CAPTIONS...")
    collection_name = "dam_objects"
    existing = [c.name for c in client.get_collections().collections]
    if collection_name not in existing:
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
        )

    jsonl_files = sorted(list(dam_dir.glob("*.jsonl")))
    total_objects = 0
    t0_start = time.time()

    for jsonl_path in tqdm(jsonl_files, desc="Embedding DAM JSONLs"):
        video_id = jsonl_path.stem
        records = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    try:
                        records.append(json.loads(line))
                    except Exception:
                        continue
        if not records:
            continue

        captions = [r.get("detailed_caption", "") or r.get("caption", "") or r.get("class_entity", "") for r in records]
        embeddings = embedder.embed_batch(captions, batch_size=batch_size)

        points = []
        for idx, (rec, vec) in enumerate(zip(records, embeddings)):
            frame_idx = int(rec.get("frame_idx", 0))
            keyframe_n = int(rec.get("keyframe_n", rec.get("n", 0)))
            region_id = rec.get("region_id", f"{video_id}:{frame_idx}:d{idx:03d}")
            class_entity = rec.get("class_entity", rec.get("class", "object"))
            bbox_norm = rec.get("bbox_norm", rec.get("bbox", [0.0, 0.0, 1.0, 1.0]))
            detailed_caption = rec.get("detailed_caption", "")
            point_id = abs(hash(f"{video_id}_{frame_idx}_{region_id}")) % (10**16)

            points.append(
                PointStruct(
                    id=point_id,
                    vector=vec.tolist(),
                    payload={
                        "video_id": video_id,
                        "frame_idx": frame_idx,
                        "keyframe_n": keyframe_n,
                        "region_id": str(region_id),
                        "class_entity": str(class_entity),
                        "bbox_norm": [float(b) for b in bbox_norm],
                        "detailed_caption": str(detailed_caption),
                    },
                )
            )

        for chunk in chunk_list(points, chunk_size=1000):
            client.upload_points(collection_name=collection_name, points=chunk, wait=True)
        total_objects += len(points)

    logger.info(f"✅ Finished DAM Indexing: {total_objects:,} objects in {time.time() - t0_start:.1f}s")

def process_asr_transcripts(asr_dir: Path, map_dir: Path, embedder: BGEFastEmbedder, client: QdrantClient, batch_size: int = 128):
    logger.info("🎙️ STEP 2: PROCESSING ASR AUDIO TRANSCRIPTS...")
    video_to_keyframes = load_map_keyframes(map_dir)
    collection_name = "keyframes"
    existing = [c.name for c in client.get_collections().collections]
    if collection_name not in existing:
        client.create_collection(
            collection_name=collection_name,
            vectors_config={
                "visual": VectorParams(size=768, distance=Distance.COSINE),
                "speech": VectorParams(size=1024, distance=Distance.COSINE),
            },
        )

    asr_files = sorted(list(asr_dir.glob("*.jsonl")) + list(asr_dir.glob("*.json")))
    total_speech_pts = 0
    t0_start = time.time()

    for asr_path in tqdm(asr_files, desc="Embedding ASR Transcripts"):
        video_id = asr_path.stem.replace(".jsonl", "").replace(".json", "")
        keyframes = video_to_keyframes.get(video_id, [])
        if not keyframes:
            continue

        segments = []
        with open(asr_path, "r", encoding="utf-8") as f:
            if asr_path.suffix == ".jsonl":
                for line in f:
                    if line.strip():
                        try:
                            segments.append(json.loads(line))
                        except Exception:
                            continue
            else:
                try:
                    data = json.load(f)
                    segments = data if isinstance(data, list) else data.get("segments", [])
                except Exception:
                    continue

        if not segments:
            continue

        kf_speech_map: dict[int, list[str]] = {}
        for seg in segments:
            start_s = float(seg.get("start_s", seg.get("start", 0.0)))
            end_s = float(seg.get("end_s", seg.get("end", 0.0)))
            text = str(seg.get("text", "")).strip()
            if not text:
                continue

            for kf in keyframes:
                pts = kf["pts_time"]
                if (start_s - 1.5) <= pts <= (end_s + 1.5):
                    f_idx = kf["frame_idx"]
                    if f_idx not in kf_speech_map:
                        kf_speech_map[f_idx] = []
                    kf_speech_map[f_idx].append(text)

        if not kf_speech_map:
            continue

        frame_indices = list(kf_speech_map.keys())
        aggregated_texts = [" ".join(kf_speech_map[f_idx]) for f_idx in frame_indices]
        embeddings = embedder.embed_batch(aggregated_texts, batch_size=batch_size)

        kf_lookup = {kf["frame_idx"]: kf for kf in keyframes}
        points = []
        for f_idx, text, vec in zip(frame_indices, aggregated_texts, embeddings):
            kf = kf_lookup[f_idx]
            point_id = abs(hash(f"{video_id}_{f_idx}")) % (10**16)

            points.append(
                PointStruct(
                    id=point_id,
                    vector={
                        "speech": vec.tolist(),
                        "visual": [0.0] * 768,
                    },
                    payload={
                        "video_id": video_id,
                        "keyframe_n": kf["n"],
                        "frame_idx": f_idx,
                        "pts_time_s": kf["pts_time"],
                        "fps": kf["fps"],
                        "speech_text": text,
                        "image_relpath": f"keyframes/{video_id}/{kf['n']}.jpg",
                    },
                )
            )

        for chunk in chunk_list(points, chunk_size=1000):
            client.upload_points(collection_name=collection_name, points=chunk, wait=True)
        total_speech_pts += len(points)

    logger.info(f"✅ Finished ASR Audio Indexing: {total_speech_pts:,} keyframes in {time.time() - t0_start:.1f}s")

if __name__ == "__main__":
    embedder = BGEFastEmbedder()
    client = QdrantClient(path="/kaggle/working/qdrant_db")
    process_dam_descriptions(Path("/kaggle/working/data/dam_descriptions"), embedder, client, batch_size=128)
    process_asr_transcripts(Path("/kaggle/working/data/dam_descriptions").parent / "asr_segments", Path("/kaggle/working/data/map-keyframes"), embedder, client, batch_size=128)
    print("\n🎉 ALL EMBEDDINGS SUCCESSFULLY CREATED!")

## 🚀 Step 5: Run Embedding Job on GPU (~10–12 minutes)

In [ ]:
!python run_kaggle_embedding.py

## 📊 Step 6: Verify Final Point Counts in Qdrant Database

In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient(path="/kaggle/working/qdrant_db")
print("=== QDRANT DATABASE SUMMARY ===")
for c in client.get_collections().collections:
    info = client.get_collection(c.name)
    print(f"  • Collection [{c.name}]: {info.points_count:,} points on disk")

## 📦 Step 7: Zip and Upload `qdrant_db.zip` Back to Google Drive

In [ ]:
import os

print("📦 Compressing /kaggle/working/qdrant_db into zip archive...")
%cd /kaggle/working
!zip -q -r qdrant_db.zip qdrant_db/

zip_size_mb = os.path.getsize("/kaggle/working/qdrant_db.zip") / (1024 * 1024)
print(f"📦 qdrant_db.zip size: {zip_size_mb:.2f} MB")

print("\n🚀 Uploading qdrant_db.zip directly to Google Drive (gdrive:AIC_HCM)...\n")
!rclone copy /kaggle/working/qdrant_db.zip "gdrive:AIC_HCM/" -P

print("\n🎉 ALL DONE! The full embedded database has been uploaded to your Google Drive.")